In [72]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [73]:
heart_dataset = pd.read_csv("C:/Documants/projects/Datasets/heart_disease_uci.csv")
heart_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    str    
 3   dataset   920 non-null    str    
 4   cp        920 non-null    str    
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    str    
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    str    
 13  ca        309 non-null    float64
 14  thal      434 non-null    str    
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 156.4+ KB


In [74]:
heart_dataset.isna().sum()

id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [75]:
# =========================================================
# 3. DROP ROWS WITH SMALL AMOUNTS OF MISSING DATA
# =========================================================
heart_dataset = heart_dataset.dropna(subset=["chol", "trestbps", "restecg", "oldpeak"])

In [76]:
heart_dataset.duplicated().sum()

np.int64(0)

In [77]:
for col in heart_dataset.columns:
    # check if column has any string values
    if heart_dataset[col].apply(lambda x: isinstance(x, str)).any():
        mask = heart_dataset[col] != heart_dataset[col].str.strip()
        
        if mask.any():
            print(f"Column '{col}' has leading/trailing spaces")

Column 'slope' has leading/trailing spaces
Column 'thal' has leading/trailing spaces


In [78]:
# =========================================================
# 4. STRIP WHITESPACE FIRST (before any mapping!)
# =========================================================
for col in ["restecg", "slope", "thal"]:
    heart_dataset[col] = heart_dataset[col].astype("string").str.strip()

for col in heart_dataset.columns:
    # check if column has any string values
    if heart_dataset[col].apply(lambda x: isinstance(x, str)).any():
        mask = heart_dataset[col] != heart_dataset[col].str.strip()
        
        if mask.any():
            print(f"Column '{col}' has leading/trailing spaces")


In [79]:
for col in heart_dataset.select_dtypes(include=["object", "string"]).columns:
    print("\n", col)
    print(heart_dataset[col].apply(lambda x: str(x).strip()).unique())


 sex
<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

 dataset
<ArrowStringArray>
['Cleveland', 'Hungary', 'Switzerland', 'VA Long Beach']
Length: 4, dtype: str

 cp
<ArrowStringArray>
['typical angina', 'asymptomatic', 'non-anginal', 'atypical angina']
Length: 4, dtype: str

 fbs
<ArrowStringArray>
['True', 'False', 'nan']
Length: 3, dtype: str

 restecg
<ArrowStringArray>
['lv hypertrophy', 'normal', 'st-t abnormality']
Length: 3, dtype: str

 exang
<ArrowStringArray>
['False', 'True']
Length: 2, dtype: str

 slope
<ArrowStringArray>
['downsloping', 'flat', 'upsloping', '<NA>']
Length: 4, dtype: str

 thal
<ArrowStringArray>
['fixed defect', 'normal', 'reversable defect', '<NA>']
Length: 4, dtype: str


In [80]:
# =========================================================
# 5. MAP CATEGORICAL TEXT COLUMNS TO NUMBERS
# (now safe, since whitespace is already gone)
# =========================================================
heart_dataset["thal"] = heart_dataset["thal"].map({
    "normal": 0,
    "fixed defect": 1,
    "reversable defect": 2
})

heart_dataset["slope"] = heart_dataset["slope"].map({
    "upsloping": 0,
    "flat": 1,
    "downsloping": 2
})

heart_dataset["sex"] = heart_dataset["sex"].map({
    "Male": 1,
    "Female": 0
})

In [81]:
# =========================================================
# 6. BINARIZE TARGET: 0 = No Disease, 1 = Disease
# =========================================================
heart_dataset["num"] = heart_dataset["num"].apply(lambda x: 0 if x == 0 else 1)

In [82]:
# =========================================================
# 7. FEATURE SELECTION
# =========================================================
selected_features = [
    'oldpeak', 'exang', 'thalch', 'age', 'sex',
    'ca', 'trestbps', 'chol', 'fbs', 'thal', 'slope'
]

X = heart_dataset[selected_features]
y = heart_dataset['num']

In [83]:
# =========================================================
# 8. TRAIN/TEST SPLIT (test set stays untouched from here on)
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert nullable pandas dtypes (Int64/Float64/boolean) to plain float64
# so sklearn can handle NaN properly
X_train = X_train.astype("float64")
X_test = X_test.astype("float64")

# =========================================================
# 9. KNN IMPUTATION (fit on train only, scaled space)
# =========================================================
scaler = StandardScaler()
X_train_scaled_temp = scaler.fit_transform(X_train)
X_test_scaled_temp = scaler.transform(X_test)

knn_imputer = KNNImputer(n_neighbors=5)
X_train_imputed = knn_imputer.fit_transform(X_train_scaled_temp)
X_test_imputed = knn_imputer.transform(X_test_scaled_temp)

# Unscale back to real values
X_train = pd.DataFrame(
    scaler.inverse_transform(X_train_imputed), columns=X_train.columns, index=X_train.index
)
X_test = pd.DataFrame(
    scaler.inverse_transform(X_test_imputed), columns=X_test.columns, index=X_test.index
)

# Round categorical columns back to valid integer categories
categorical_cols = ["fbs", "ca", "slope", "thal", "exang"]
for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].round().astype(int)
        X_test[col] = X_test[col].round().astype(int)

# Re-scale final clean version (needed for Logistic Regression)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# =========================================================
# 10. GRIDSEARCHCV WITH SMOTE INSIDE THE PIPELINE
# =========================================================

# --- Random Forest ---
rf_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42, class_weight="balanced"))
])
rf_params = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring="accuracy", n_jobs=-1)
rf_grid.fit(X_train, y_train)

# --- Gradient Boosting ---
gb_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("gb", GradientBoostingClassifier(random_state=42))
])
gb_params = {
    "gb__n_estimators": [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1],
    "gb__max_depth": [2, 3, 4],
    "gb__min_samples_split": [2, 5, 10]
}
gb_grid = GridSearchCV(gb_pipeline, gb_params, cv=5, scoring="accuracy", n_jobs=-1)
gb_grid.fit(X_train, y_train)

# --- Logistic Regression ---
lr_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("lr", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])
lr_params = {
    "lr__C": [0.01, 0.1, 1, 10, 100]
}
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring="accuracy", n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)

# =========================================================
# 11. REPORT CV RESULTS (the honest, leakage-free numbers)
# =========================================================
print("==============================")
print("Best CV Scores (leakage-free)")
print("==============================")
print(f"Random Forest:       {rf_grid.best_score_:.4f}  | {rf_grid.best_params_}")
print(f"Gradient Boosting:   {gb_grid.best_score_:.4f}  | {gb_grid.best_params_}")
print(f"Logistic Regression: {lr_grid.best_score_:.4f}  | {lr_grid.best_params_}")


Best CV Scores (leakage-free)
Random Forest:       0.8227  | {'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 10, 'rf__n_estimators': 300}
Gradient Boosting:   0.8318  | {'gb__learning_rate': 0.05, 'gb__max_depth': 2, 'gb__min_samples_split': 2, 'gb__n_estimators': 100}
Logistic Regression: 0.8167  | {'lr__C': 0.1}


In [85]:

# =========================================================
# 12. FINAL EVALUATION ON THE UNTOUCHED TEST SET (Gradient Boosting)
# =========================================================
final_model = gb_grid.best_estimator_
final_model.fit(X_train, y_train)   # pipeline applies SMOTE internally
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

print("\n==============================")
print("Final Evaluation - Test Set (Gradient Boosting)")
print("==============================")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted")
print("\nOverall performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


Final Evaluation - Test Set (Gradient Boosting)

Classification report:
              precision    recall  f1-score   support

  No Disease       0.79      0.76      0.77        74
     Disease       0.81      0.84      0.82        91

    accuracy                           0.80       165
   macro avg       0.80      0.80      0.80       165
weighted avg       0.80      0.80      0.80       165


Confusion matrix:
[[56 18]
 [15 76]]

Overall performance:
Accuracy:  0.8000
Precision: 0.7996
Recall:    0.8000
F1-score:  0.7996


In [86]:
from sklearn.model_selection import cross_val_score

cv10_scores = cross_val_score(gb_grid.best_estimator_, X_train, y_train, cv=10, scoring="accuracy")
print(f"10-Fold CV Accuracy: {cv10_scores.mean():.4f} (+/- {cv10_scores.std():.4f})")

10-Fold CV Accuracy: 0.8242 (+/- 0.0424)
